# Evaluate honesty

Run in Google Colab. Configure the repo URL, commit, RUN_ROOT, and model names below. GPU stages require a CUDA-enabled runtime. Keep RUN_ROOT and all evaluation settings identical across base/peer/control. API secrets stay in environment variables. Each benchmark runs in a separate process so GPU memory is released.


In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/seungjun-green/sycophancy-honest-peer-finetuning"  # Cloned by a fresh Colab runtime.
GIT_REF = "main"  # Use a commit SHA for a reproducible experiment.
PROJECT_DIR = Path.cwd() if Path("src/honest_peer/cli.py").exists() else Path.cwd() / "honest-peer"
RUN_ROOT = PROJECT_DIR / "outputs"  # Local working files; Google Drive snapshots provide persistence.
STORAGE_MODE = "google_drive"  # Hosted Colab default; use "drive_directory" for an existing local Drive mount/sync.
DRIVE_SUBDIR = "honest-peer-experiment"  # MyDrive subfolder for hosted Colab backups.
DRIVE_BACKUP_ROOT = None  # Local runtime example: Path("/your/mounted-or-synced-drive/honest-peer-experiment")
BACKUP_INTERVAL_SECONDS = 60
CONFIG = "configs/default.yaml"  # Eval: configs/peer.yaml or configs/control.yaml. Control train: configs/control.yaml.
OVERRIDES = []  # e.g. ["rewrite.model=ACCOUNT_VISIBLE_MODEL", "judge.model=ACCOUNT_VISIBLE_MODEL"]
LIVE_SMOKE = False  # Small real API/GPU run. smoke_test.ipynb uses synthetic data and runs on CPU.


In [ ]:
if not (PROJECT_DIR / "pyproject.toml").exists():
    if not REPO_URL:
        raise ValueError("Set REPO_URL to your GitHub repository URL.")
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", GIT_REF], check=True)
os.chdir(PROJECT_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "uv==0.11.22"], check=True)
EXTRAS = ['eval']
subprocess.run(["uv", "sync", "--frozen", *[arg for extra in EXTRAS for arg in ("--extra", extra)]], check=True)
RUN_ROOT.mkdir(parents=True, exist_ok=True)


## Save intermediate outputs to Google Drive

Google Drive backup is enabled by default for hosted Colab (STORAGE_MODE = google_drive). Run the storage cell to mount Drive and authorize access. Local runtimes: choose drive_directory and supply a folder already mounted or synced on the local machine; Colab's built-in Drive mount is unavailable there. Outputs and active SQLite databases stay in local RUN_ROOT. A consistent snapshot is saved to Drive every BACKUP_INTERVAL_SECONDS and when each stage exits. An empty RUN_ROOT is restored from the latest snapshot. Keep the same experiment folder and one writer at a time. An abrupt runtime loss can redo work since the last completed snapshot. [Colab storage/local-runtime FAQ](https://research.google.com/colaboratory/faq.html)


In [ ]:
# Only standard-library code is imported into the notebook kernel.
sys.path.insert(0, str(PROJECT_DIR / "src"))
from honest_peer.storage import restore_outputs, run_with_backup

if STORAGE_MODE == "google_drive":
    from google.colab import drive
    drive.mount("/content/drive")
    BACKUP_ROOT = Path("/content/drive/MyDrive") / DRIVE_SUBDIR
elif STORAGE_MODE == "drive_directory":
    if DRIVE_BACKUP_ROOT is None or not Path(DRIVE_BACKUP_ROOT).parent.is_dir():
        raise ValueError("Set DRIVE_BACKUP_ROOT inside an existing mounted or synced Google Drive directory.")
    BACKUP_ROOT = Path(DRIVE_BACKUP_ROOT)
elif STORAGE_MODE == "local":
    BACKUP_ROOT = None
else:
    raise ValueError("Choose local, google_drive, or drive_directory for STORAGE_MODE.")

if BACKUP_ROOT is not None and not any(RUN_ROOT.iterdir()):
    print(restore_outputs(RUN_ROOT, BACKUP_ROOT))
print("Working outputs:", RUN_ROOT)
print("Drive backups:", BACKUP_ROOT)

def run_stage(args, capture_output=False):
    # Colab's inline plotting backend is unavailable in the isolated project environment.
    stage_env = {**os.environ, "MPLBACKEND": "Agg"}
    return run_with_backup(args, RUN_ROOT, BACKUP_ROOT,
                           interval_seconds=BACKUP_INTERVAL_SECONDS,
                           capture_output=capture_output, env=stage_env)


## API keys from Colab Secrets

Add ANTHROPIC_API_KEY (or CLAUDE_API_KEY) and HF_TOKEN in Colab's Secrets panel (key icon in the left sidebar) and enable Notebook access. The next cell reads them with userdata.get and passes them to the project subprocess through the environment. Nothing is typed, printed, or written to disk.


In [ ]:
import os
# Keys come from Colab Secrets (key icon in the left sidebar; enable Notebook access).
# ANTHROPIC_API_KEY may also be stored under the name CLAUDE_API_KEY.
SECRET_NAMES = {"ANTHROPIC_API_KEY": ["ANTHROPIC_API_KEY", "CLAUDE_API_KEY"], "HF_TOKEN": ["HF_TOKEN"]}
try:
    from google.colab import userdata
except ImportError:
    userdata = None  # Not in Colab: export the variables in your shell instead.

for key in ['ANTHROPIC_API_KEY', 'HF_TOKEN']:
    if os.environ.get(key):
        print(key, "already set in the environment.")
        continue
    names = SECRET_NAMES.get(key, [key])
    for name in names if userdata is not None else []:
        try:
            value = userdata.get(name)
        except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
            continue
        if value:
            os.environ[key] = value
            print(key, "loaded from Colab Secrets (" + name + ").")
            break
    if not os.environ.get(key):
        raise RuntimeError(
            "Add " + " or ".join(names) + " to Colab Secrets and enable Notebook access, then rerun this cell."
            if userdata is not None
            else "Set " + key + " in the environment before starting Jupyter."
        )
value = None


In [ ]:
import json
PYTHON = str(PROJECT_DIR / ".venv/bin/python")
CLI = str(PROJECT_DIR / ".venv/bin/honest-peer")
# Resolve portable package prompts in the repo's locked environment.
script = """import json,sys
from pathlib import Path
from honest_peer.config import load_config
cfg=load_config(sys.argv[1],json.loads(sys.argv[2]))
root=Path(sys.argv[3]).resolve()
cfg['paths'].update({key:str(root/key) for key in ('data','prepared','evaluation','comparison','cache')})
variant={'original':'control','rewritten':'peer'}.get(cfg['training']['data_variant'],cfg['training']['data_variant'])
cfg['paths']['training']=str(root/'training'/variant)
if cfg['model']['adapter']:
    cfg['model']['adapter']=str(root/'training'/cfg['model']['variant']/'adapter')
print(json.dumps(cfg))"""
effective = subprocess.check_output([PYTHON, "-c", script, CONFIG, json.dumps(OVERRIDES), str(RUN_ROOT)], text=True)
effective_path = RUN_ROOT / ("config-" + Path(CONFIG).stem + ".json")
effective_path.write_text(effective)


In [ ]:
COMMANDS = [('evaluate', ['--benchmark', 'simpleqa']), ('evaluate', ['--benchmark', 'mask'])]
for stage, options in COMMANDS:
    args = [CLI, stage, "--config", str(effective_path), *options]
    if LIVE_SMOKE and stage != "smoke":
        args.append("--smoke")
    run_stage(args)
